In [112]:
import pandas as pd
import numpy as np

In [113]:
df=pd.read_csv("car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [114]:
df.T

,0,1,2,3,4,5,6,7,8,9,...,9990,9991,9992,9993,9994,9995,9996,9997,9998,9999
model_year,2006,2008,1996,1989,1994,2007,1992,2005,1997,2004,...,1975,1987,2000,2007,1993,2005,1993,2014,2004,2011
origin,Europe,Europe,Asia,Europe,USA,USA,Asia,USA,Asia,USA,...,Asia,USA,Europe,USA,Asia,USA,Asia,USA,USA,USA
fuel_type,Gasoline,Diesel,Gasoline,Gasoline,Diesel,Hybrid,Gasoline,Gasoline,Gasoline,Diesel,...,Diesel,Gasoline,Gasoline,Diesel,Gasoline,Gasoline,Gasoline,Gasoline,Hybrid,Diesel
drivetrain,Front-wheel drive,Front-wheel drive,Front-wheel drive,Front-wheel drive,Front-wheel drive,Front-wheel drive,Front-wheel drive,Front-wheel drive,Rear-wheel drive,All-wheel drive,...,Front-wheel drive,Rear-wheel drive,Front-wheel drive,Rear-wheel drive,Rear-wheel drive,All-wheel drive,All-wheel drive,All-wheel drive,Rear-wheel drive,Front-wheel drive
num_doors,4,4,5,4,3,2,2,4,3,5,...,4,4,3,2,2,4,2,3,5,4
engine_displacement,2180,2390,2320,2130,2580,2290,2270,2370,2330,2330,...,1940,2130,2470,1950,2360,2470,2300,2480,2180,2380
num_cylinders,6,6,6,6,7,6,6,6,6,6,...,6,5,7,6,6,6,6,6,6,6
horsepower,243.0,272.0,267.0,258.0,304.0,266.0,250.0,279.0,251.0,280.0,...,236.0,228.0,271.0,234.0,257.0,271.0,244.0,267.0,270.0,295.0
vehicle_weight,3870,4210,4240,4490,4510,4260,4310,4090,4230,4620,...,3630,4040,3990,4250,4610,4690,4370,4590,4450,4370
acceleration,NaN,NaN,17.3,18.7,17.5,18.8,21.7,17.4,19.5,17.7,...,19.7,17.2,17.3,19.9,22.2,18.9,19.0,18.4,20.9,17.6


In [115]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [116]:
df.dtypes

model_year               int64
origin                  object
fuel_type               object
drivetrain              object
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [117]:
for v in ['horsepower','acceleration']:
    df[v]=df[v].fillna(0)

In [118]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [119]:
from sklearn.model_selection import train_test_split

In [120]:
df_full_train, df_test =train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val =train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val=df_val['fuel_efficiency_mpg'].values
y_test=df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']
len(df_train),len(df_test),len(df_val)
len(y_train)

6000

In [121]:
from sklearn.feature_extraction import DictVectorizer

In [122]:
train_dicts=df_train.to_dict(orient='records')
val_dicts=df_val.to_dict(orient='records')
test_dicts=df_test.to_dict(orient='records')

dv=DictVectorizer(sparse=True)

x_train=dv.fit_transform(train_dicts)
x_val=dv.transform(val_dicts)
x_test=dv.transform(test_dicts)

In [123]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import  export_text

In [124]:
dt=DecisionTreeRegressor(max_depth=1)
dt.fit(x_train,y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [125]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score

In [128]:
rf = RandomForestRegressor(n_estimators=10,random_state=1,n_jobs=-1)
rf.fit(x_train,y_train)

y_pred=rf.predict(x_val)
rmse=np.sqrt(np.mean((y_val-y_pred)**2))
rmse


np.float64(1.837054054730018)

In [ ]:
scores=[]
for v in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=v,random_state=1,n_jobs=-1)
    rf.fit(x_train,y_train)

    y_pred=rf.predict(x_val)
    rmse=np.sqrt(np.mean((y_val-y_pred)**2))
    scores.append((v,round(rmse,3)))
        

In [135]:
df_scores=pd.DataFrame(scores,columns=['n_estimators','RMSE'])

In [136]:
df_scores.sort_values('RMSE',ascending=False)

,n_estimators,RMSE
0,10,1.837
1,50,1.772
3,150,1.769
2,100,1.768


In [138]:
scores=[]
for max in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n,max_depth=max,random_state=1,n_jobs=-1)
        rf.fit(x_train,y_train)
        
        y_pred=rf.predict(x_val)
        rmse=np.sqrt(np.mean((y_val-y_pred)**2))
        scores.append((max,n,round(rmse,3)))
        

In [143]:
columns=['max_depth', 'n_estimators', 'rmse']
df_scores = pd.DataFrame(scores, columns=columns)
df_scores.groupby('max_depth').rmse.mean()

max_depth
10    1.75575
15    1.78400
20    1.78825
25    1.78625
Name: rmse, dtype: float64

In [144]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [145]:
import xgboost as xgb

In [148]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(x_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(x_val, label=y_val, feature_names=features)

In [149]:
watchlist=[(dtrain, 'train'),(dval, 'val')]

In [ ]:

xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
evals_res={}
model = xgb.train(xgb_params, dtrain, num_boost_round=100,evals=watchlist,evals_result=evals_res)
val_rmse = evals_res['val']['rmse']
print(f"eta=0.3:RMSE val final: {val_rmse[-1]:.4f}")


[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va

In [160]:
xgb_params = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
evals_res={}
model = xgb.train(xgb_params, dtrain, num_boost_round=100,evals=watchlist,evals_result=evals_res)
val_rmse = evals_res['val']['rmse']
print(f"eta=0.1:RMSE val final: {val_rmse[-1]:.4f}")

[0]	train-rmse:2.74739	val-rmse:2.78635
[1]	train-rmse:2.58527	val-rmse:2.63162
[2]	train-rmse:2.44540	val-rmse:2.50036
[3]	train-rmse:2.32187	val-rmse:2.38568
[4]	train-rmse:2.21445	val-rmse:2.28765
[5]	train-rmse:2.12109	val-rmse:2.20428
[6]	train-rmse:2.03996	val-rmse:2.13241
[7]	train-rmse:1.96795	val-rmse:2.06798
[8]	train-rmse:1.90640	val-rmse:2.01618
[9]	train-rmse:1.85240	val-rmse:1.96998
[10]	train-rmse:1.80592	val-rmse:1.93152
[11]	train-rmse:1.76410	val-rmse:1.89688
[12]	train-rmse:1.72781	val-rmse:1.86726
[13]	train-rmse:1.69564	val-rmse:1.84312
[14]	train-rmse:1.66745	val-rmse:1.82218
[15]	train-rmse:1.64286	val-rmse:1.80346
[16]	train-rmse:1.62187	val-rmse:1.79038
[17]	train-rmse:1.60321	val-rmse:1.77726
[18]	train-rmse:1.58694	val-rmse:1.76827
[19]	train-rmse:1.57240	val-rmse:1.75951
[20]	train-rmse:1.55847	val-rmse:1.75076
[21]	train-rmse:1.54640	val-rmse:1.74347
[22]	train-rmse:1.53643	val-rmse:1.73912
[23]	train-rmse:1.52675	val-rmse:1.73485
[24]	train-rmse:1.51763	va

In [ ]:
eta=0.3:RMSE val final: 1.8265 
eta=0.1:RMSE val final: 1.7248 